# SO-ARM101 押印ロボット：お手本データの記録

シミュレーションの「先生役」に押印させ、その様子を **カメラ映像 + 関節の角度** として記録し、LeRobot の学習データ（データセット）にします。

**使い方**
1. メニューの「ランタイム」→「ランタイムのタイプを変更」で、使えるなら **T4 GPU** を選ぶ。GPU の使用量上限に達しているときは **CPU** のままで大丈夫です（描画が遅くなるだけで、結果は同じです）
2. 上のセルから順番に実行する（「ランタイム」→「すべてのセルを実行」でもOK）
3. お手本は1回ずつ Google ドライブに保存されるので、途中で止まっても「8」をもう一度実行すれば続きから再開できる

**全体の流れ**

| 番号 | やること |
|---|---|
| 1 | ライブラリのインストール |
| 2 | 設定 |
| 3 | アームにハンコを持たせたモデルを作る |
| 4 | 動かすための道具（逆運動学・経路など） |
| 5 | カメラの位置を確認 |
| 6 | お手本1回分を記録する仕組み |
| 7 | 1回だけ試して動画で確認 |
| 8 | たくさん記録する（1回ずつドライブに保存、途中から再開できる） |
| 9 | 保存したお手本からデータセットを作る |
| 10 | データセットの中身を確認 |

## 1. ライブラリのインストール

- `lerobot[dataset]==0.6.1`：学習データの形式（LeRobotDataset）を作るためのライブラリ。学習で使うバージョンとそろえています。
- `mujoco`：物理シミュレーション
- `mediapy`：動画の表示と保存
- `SO-ARM100` リポジトリ：SO101 アームのモデル（`scene.xml`）が入っています

数分かかります。終わったあとに「セッションを再起動」を求める表示が出たら、再起動してから **次のセル（2. 設定）から** 実行してください。

In [ ]:
!pip install -q "lerobot[dataset]==0.6.1" mujoco mediapy
# GPU が無いとき（CPU のランタイム）は、ソフトウェアで描画する部品を入れる
!nvidia-smi > /dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y libosmesa6 > /dev/null)
![ -d /content/SO-ARM100 ] || git clone -q https://github.com/TheRobotStudio/SO-ARM100.git /content/SO-ARM100
print("インストール完了")

## 2. 設定

あとで変えたくなる数字を全部ここに集めています。数字を変えたいときは、このセルだけ書き換えて、ここから下を実行し直してください。

- **ハンコ・成功条件・動き方**：前のノートブックで決めた値です（押し付けの深さは 1mm）。
- **`N_EPISODES`**：記録するお手本の回数。ACT の学習では 50 回くらいが最初の目安です。
- **`IMG_H, IMG_W`**：カメラ画像の大きさ。実機の Logicool カメラを 640×480 に縮めて使う想定に合わせています。
- **`DEMO_AIM_NOISE_MM`**：お手本の狙いにわざと入れる誤差。先生役は正確に押すべきなので 0 にしています。
- **`TASK`**：AI に渡す指示の文。SmolVLA のように言葉を読むモデルのために英語で入れています（ACT は使いません）。

In [ ]:
import os
import subprocess

# GPU があれば GPU で、無ければ CPU（ソフトウェア）で画像を描く
HAS_GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
os.environ["MUJOCO_GL"] = "egl" if HAS_GPU else "osmesa"
os.environ["PYOPENGL_PLATFORM"] = os.environ["MUJOCO_GL"]
print("描画:", "GPU" if HAS_GPU else "CPU（GPU なし。少し遅くなります）")

import csv
import shutil
from pathlib import Path

import numpy as np
import mujoco
import mediapy as media
import matplotlib.pyplot as plt

# ---- モデル ----
XML_PATH = "/content/SO-ARM100/Simulation/SO101/scene.xml"

# ---- ハンコ ----
STAMP_BODY_DIAMETER = 0.0105    # 胴体の直径 [m]
STAMP_LENGTH = 0.062            # 全長 [m]
STAMP_PROTRUDE = 0.025          # 指先の目印から印面までの距離 [m]

# ---- 成功条件 ----
STAMP_DIAMETER_MM = 10.0        # 印影の直径
FRAME_INNER_DIAMETER_MM = 20.0  # 丸枠の内径
MAX_OFFSET_MM = (FRAME_INNER_DIAMETER_MM - STAMP_DIAMETER_MM) / 2  # 許容ずれ 5mm

# ---- 動き方 ----
PAPER_Z = 0.0                   # 紙の表面の高さ [m]
APPROACH_HEIGHT = 0.04          # 枠の真上で一度止まる高さ [m]
PRESS_DEPTH = 0.001             # 紙の面より少し下を狙って押し付ける [m]
DESCENT_STEP_MM = 1.0           # まっすぐ下りる経路を何mmごとに計算するか
REACH_RANGE = (0.16, 0.26)      # 紙を置く距離の範囲 [m]
ANGLE_RANGE_DEG = 25            # 紙を置く向きの範囲（正面から左右）[度]
DEMO_AIM_NOISE_MM = 0.0         # お手本の狙いに入れる誤差

# ---- 記録 ----
N_EPISODES = 50                 # 保存するお手本の回数
MAX_ATTEMPTS = 80               # 失敗も含めた最大の試行回数
fps = 30                        # 1秒あたりの記録枚数
IMG_H, IMG_W = 480, 640         # カメラ画像の大きさ
SEED = 0                        # 乱数の種（同じ値なら毎回同じ紙の位置になる）
REPO_ID = "local/so101_stamp_sim"
DATASET_ROOT = Path("/content/so101_stamp_sim")
TASK = "Press the stamp inside the circle on the paper."

# ---- カメラ（机に固定したカメラのつもり）----
CAM_LOOKAT = [0.25, 0.0, 0.0]   # カメラが見つめる点 [m]
CAM_DISTANCE = 0.45             # その点からの距離 [m]
CAM_AZIMUTH = 180               # 水平方向の向き [度]
CAM_ELEVATION = -55             # 見下ろす角度 [度]

ARM_JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]
rng = np.random.default_rng(SEED)
print("設定完了")

## 3. アームにハンコを持たせたモデルを作る

前のノートブックと同じ処理です。

1. 元の `scene.xml` を読み込み、初期姿勢で「手首から指先へ向かう向き」を調べて、ハンコの軸にします。
2. 元の XML を設計図（`MjSpec`）として読み直し、グリッパーに長さ 62mm の円柱（ハンコ）と、印面の目印（`stamp_face`）を付け足します。
3. 設計図からモデルを作り直します。

印面は指先の目印より 25mm 先にあるので、押したときに指より先に印面が紙に当たります。

In [ ]:
base_model = mujoco.MjModel.from_xml_path(XML_PATH)
base_data = mujoco.MjData(base_model)
if base_model.nkey > 0:
    mujoco.mj_resetDataKeyframe(base_model, base_data, 0)
mujoco.mj_forward(base_model, base_data)

frame_id = mujoco.mj_name2id(base_model, mujoco.mjtObj.mjOBJ_SITE, "gripperframe")
holder_id = int(base_model.site_bodyid[frame_id])
holder_name = base_model.body(holder_id).name
parent_id = int(base_model.body_parentid[holder_id])

# ハンコの軸：手首から指先へ向かう向き（グリッパーのボディ座標）
v_world = base_data.site_xpos[frame_id] - base_data.xpos[parent_id]
R_holder = base_data.xmat[holder_id].reshape(3, 3)
axis_body = R_holder.T @ v_world
axis_body /= np.linalg.norm(axis_body)

frame_pos = base_model.site_pos[frame_id].copy()
face_pos = frame_pos + axis_body * STAMP_PROTRUDE        # 印面の中心
center_pos = face_pos - axis_body * STAMP_LENGTH / 2     # 円柱の中心
stamp_quat = np.zeros(4)
mujoco.mju_quatZ2Vec(stamp_quat, axis_body)              # 円柱のz軸をハンコの軸に合わせる

spec = mujoco.MjSpec.from_file(XML_PATH)
holder = spec.body(holder_name) if hasattr(spec, "body") else spec.find_body(holder_name)

stamp = holder.add_geom()
stamp.name = "stamp"
stamp.type = mujoco.mjtGeom.mjGEOM_CYLINDER
stamp.size = [STAMP_BODY_DIAMETER / 2, STAMP_LENGTH / 2, 0]
stamp.pos = center_pos
stamp.quat = stamp_quat
stamp.mass = 0.01
stamp.rgba = [0.1, 0.1, 0.1, 1]

face = holder.add_site()
face.name = "stamp_face"
face.pos = face_pos
face.quat = stamp_quat
face.size = [0.002, 0, 0]

sim_model = spec.compile()
sim_data = mujoco.MjData(sim_model)
print(f"ハンコを '{holder_name}' に持たせました")

## 4. 動かすための道具

前のノートブックで作った道具をまとめています。

- **`reset_pose()`**：初期姿勢に戻し、全モーターの目標を今の角度に合わせます（急に動き出さないように）。
- **`tip_pose()`**：印面の位置と向きを返します。
- **モーターの一覧**：記録には腕の5関節とグリッパーの全部（6個）を使います。`MOTOR_NAMES` は実機の LeRobot と同じ `shoulder_pan.pos` のような名前にしています。
- **`solve_ik()`（逆運動学）**：「印面をここに置きたい」から「各関節の角度」を逆算します。
- **`vertical_path()`**：1mm ごとに逆運動学を解き、真下へまっすぐ下りる経路を作ります。
- **`pose_on_path()`**：経路の何割まで進んだかを受け取り、その位置の関節角度を返します。動き始めと止まる直前はゆっくりです。
- **`judge_stamp()`**：印影の中心が枠の中心から 5mm 以内なら成功です。
- **`draw_overlay()`**：画像に白い紙、赤い丸枠、押した後の印影を描き足します。シミュレーションの床には紙が無いので、カメラ画像にだけ描きます。
- **`sample_frame_center()`**：アームが届く範囲に、紙（丸枠）の位置をランダムに決めます。

In [ ]:
def name_to_id(obj, name):
    return mujoco.mj_name2id(sim_model, obj, name)

def reset_pose():
    # 初期姿勢に戻し、全モーターの目標を今の角度に合わせる
    if sim_model.nkey > 0:
        mujoco.mj_resetDataKeyframe(sim_model, sim_data, 0)
    else:
        mujoco.mj_resetData(sim_model, sim_data)
    mujoco.mj_forward(sim_model, sim_data)
    for i in range(sim_model.nu):
        jid = int(sim_model.actuator_trnid[i, 0])
        sim_data.ctrl[i] = sim_data.qpos[sim_model.jnt_qposadr[jid]]

# ---- 印面 ----
tip_site_id = name_to_id(mujoco.mjtObj.mjOBJ_SITE, "stamp_face")
tip_body_id = int(sim_model.site_bodyid[tip_site_id])
TOOL_AXIS_LOCAL = np.array([0.0, 0.0, 1.0])  # 印面の目印のz軸 = ハンコの軸
DOWN = np.array([0.0, 0.0, -1.0])

def tip_pose(data):
    return (data.site_xpos[tip_site_id].copy(),
            data.site_xmat[tip_site_id].reshape(3, 3).copy())

# ---- 腕の関節（逆運動学で動かすもの）----
arm = []
for name in ARM_JOINTS:
    jid = name_to_id(mujoco.mjtObj.mjOBJ_JOINT, name)
    acts = [i for i in range(sim_model.nu) if int(sim_model.actuator_trnid[i, 0]) == jid]
    arm.append({"name": name, "jid": jid,
                "qadr": int(sim_model.jnt_qposadr[jid]),
                "dadr": int(sim_model.jnt_dofadr[jid]),
                "act": acts[0] if acts else None})
qadr = np.array([j["qadr"] for j in arm])
dadr = np.array([j["dadr"] for j in arm])

def set_arm_ctrl(q_cmd):
    # 腕のモーターに目標角度を送る（指令範囲内に収める）
    for i, j in enumerate(arm):
        if j["act"] is None:
            continue
        c = q_cmd[i]
        if sim_model.actuator_ctrllimited[j["act"]]:
            c = np.clip(c, *sim_model.actuator_ctrlrange[j["act"]])
        sim_data.ctrl[j["act"]] = c

# ---- 記録する全モーター（腕5つ + グリッパー）----
MOTOR_NAMES, motor_qadr = [], []
for i in range(sim_model.nu):
    jid = int(sim_model.actuator_trnid[i, 0])
    MOTOR_NAMES.append(f"{sim_model.joint(jid).name}.pos")
    motor_qadr.append(int(sim_model.jnt_qposadr[jid]))
motor_qadr = np.array(motor_qadr)
print("記録するモーター:", MOTOR_NAMES)

# ---- アームの根元と正面の向き ----
reset_pose()
tip_home, _ = tip_pose(sim_data)
base_id = int(sim_model.jnt_bodyid[arm[0]["jid"]])
base_xy = sim_data.xpos[base_id][:2].copy()
d = tip_home[:2] - base_xy
forward = float(np.arctan2(d[1], d[0])) if np.linalg.norm(d) > 0.02 else 0.0
print(f"根元 {base_xy.round(3)}、正面の向き {np.degrees(forward):.1f}°")

# ---- 逆運動学 ----
ik_data = mujoco.MjData(sim_model)

def skew(a):
    return np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])

def solve_ik(target, q_start, iters=300, axis_weight=0.05, damping=1e-4):
    ik_data.qpos[:] = q_start
    for _ in range(iters):
        mujoco.mj_kinematics(sim_model, ik_data)
        mujoco.mj_comPos(sim_model, ik_data)
        p, R = tip_pose(ik_data)
        axis = R @ TOOL_AXIS_LOCAL
        if np.linalg.norm(p - target) < 2e-4 and np.linalg.norm(axis - DOWN) < 0.02:
            break
        jacp = np.zeros((3, sim_model.nv))
        jacr = np.zeros((3, sim_model.nv))
        mujoco.mj_jac(sim_model, ik_data, jacp, jacr, p, tip_body_id)
        J = np.vstack([jacp[:, dadr], axis_weight * (-skew(axis) @ jacr[:, dadr])])
        r = np.concatenate([p - target, axis_weight * (axis - DOWN)])
        dq = -J.T @ np.linalg.solve(J @ J.T + damping * np.eye(6), r)
        ik_data.qpos[qadr] += np.clip(dq, -0.1, 0.1)
        for j in arm:
            if sim_model.jnt_limited[j["jid"]]:
                lo, hi = sim_model.jnt_range[j["jid"]]
                ik_data.qpos[j["qadr"]] = np.clip(ik_data.qpos[j["qadr"]], lo, hi)
    mujoco.mj_kinematics(sim_model, ik_data)
    p, R = tip_pose(ik_data)
    tilt_deg = np.degrees(np.arccos(np.clip(np.dot(R @ TOOL_AXIS_LOCAL, DOWN), -1, 1)))
    return ik_data.qpos[qadr].copy(), float(np.linalg.norm(p - target)), float(tilt_deg)

# ---- 経路 ----
def smoothstep(x):
    return 3 * x**2 - 2 * x**3

def vertical_path(aim, z_top, z_bottom, q_start_full):
    # 枠の真上 z_top から z_bottom まで、真下へ一直線に下りる関節角度の列
    n = max(2, int(abs(z_top - z_bottom) * 1000 / DESCENT_STEP_MM) + 1)
    q_full = q_start_full.copy()
    path, worst_err = [], 0.0
    for z in np.linspace(z_top, z_bottom, n):
        q_arm, err, _ = solve_ik(np.array([aim[0], aim[1], z]), q_full, iters=100)
        q_full[qadr] = q_arm  # 1つ上の点の答えから解き始める
        path.append(q_arm)
        worst_err = max(worst_err, err)
    return np.array(path), worst_err

def pose_on_path(path, s):
    # 経路上の位置 s（0〜1）での関節角度
    if len(path) == 1:
        return path[0]
    x = smoothstep(float(np.clip(s, 0, 1))) * (len(path) - 1)
    i = min(int(x), len(path) - 2)
    return path[i] + (path[i + 1] - path[i]) * (x - i)

# ---- 判定 ----
def judge_stamp(stamp_center_xy_mm, frame_center_xy_mm):
    offset = float(np.linalg.norm(np.asarray(stamp_center_xy_mm) - np.asarray(frame_center_xy_mm)))
    return {"success": offset <= MAX_OFFSET_MM, "offset_mm": offset}

# ---- 画像への描き足し ----
def add_shape(scene, gtype, size, pos, rgba):
    if scene.ngeom >= scene.maxgeom:
        return
    mujoco.mjv_initGeom(scene.geoms[scene.ngeom], gtype,
                        np.asarray(size, float), np.asarray(pos, float),
                        np.eye(3).flatten(), np.asarray(rgba, np.float32))
    scene.ngeom += 1

def draw_overlay(scene, frame_center, stamp_xy):
    x, y = frame_center[:2]
    CYL, BOX = mujoco.mjtGeom.mjGEOM_CYLINDER, mujoco.mjtGeom.mjGEOM_BOX
    add_shape(scene, BOX, [0.074, 0.0525, 0.0002], [x, y, PAPER_Z + 0.0002], [1, 1, 1, 1])     # 紙
    add_shape(scene, CYL, [0.011, 0.0002, 0], [x, y, PAPER_Z + 0.0006], [0.85, 0.1, 0.1, 1])   # 枠の線
    add_shape(scene, CYL, [0.010, 0.0002, 0], [x, y, PAPER_Z + 0.0008], [1, 1, 1, 1])          # 枠の内側
    if stamp_xy is not None:
        add_shape(scene, CYL, [STAMP_DIAMETER_MM / 2000, 0.0002, 0],
                  [stamp_xy[0], stamp_xy[1], PAPER_Z + 0.0010], [0.8, 0.0, 0.0, 0.8])     # 印影

# ---- 紙の位置 ----
def sample_frame_center():
    # アームが届く範囲に、紙（丸枠）の位置をランダムに決める
    reset_pose()
    q_start = sim_data.qpos.copy()
    for _ in range(50):
        r = rng.uniform(*REACH_RANGE)
        a = forward + np.radians(rng.uniform(-ANGLE_RANGE_DEG, ANGLE_RANGE_DEG))
        c = np.array([base_xy[0] + r * np.cos(a), base_xy[1] + r * np.sin(a), PAPER_Z])
        _, err, _ = solve_ik(c + [0, 0, -PRESS_DEPTH], q_start)
        if err < 1e-3:
            return c
    raise RuntimeError("届く紙の位置が見つかりません。REACH_RANGE を見直してください")

print("道具の準備完了")

## 5. カメラの位置を確認

机に固定したカメラのつもりで、アームの正面やや上から作業エリアを見下ろす視点を作ります。紙を置いた状態の画像を1枚表示するので、次の点を確認してください。

- 白い紙と赤い丸枠が、画像の中にはっきり写っているか
- アームが紙を隠していないか

見えにくい場合は、「2. 設定」の `CAM_` で始まる数字を変えて、2 から実行し直します。

- `CAM_DISTANCE` を小さくすると近づく（枠が大きく写る）
- `CAM_ELEVATION` を -90 に近づけると真上からの視点になる
- `CAM_AZIMUTH` を変えると、カメラが机のまわりを回る

In [ ]:
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat[:] = CAM_LOOKAT
cam.distance = CAM_DISTANCE
cam.azimuth = CAM_AZIMUTH
cam.elevation = CAM_ELEVATION

preview_center = sample_frame_center()
reset_pose()
with mujoco.Renderer(sim_model, height=IMG_H, width=IMG_W) as renderer:
    renderer.update_scene(sim_data, camera=cam)
    draw_overlay(renderer.scene, preview_center, None)
    preview = renderer.render().copy()
media.show_image(preview, title="カメラの視点")

## 6. お手本1回分を記録する仕組み

### 動きの予定表（`plan_episode`）
1回のお手本は、次の7つの区間でできています（合計 4.8 秒 = 144 枚）。

| 区間 | 秒 | 内容 |
|---|---|---|
| 待つ | 0.5 | 紙が置かれた直後、初期姿勢のまま |
| 移動 | 1.2 | 枠の真上 4cm へ |
| 押す | 0.8 | 真下へまっすぐ下りて押す |
| 押したまま | 0.4 | そのまま待つ |
| 持ち上げる | 0.6 | 真上へまっすぐ戻る |
| 戻る | 1.0 | 初期姿勢へ |
| 待つ | 0.3 | 動きの終わり |

最初に「待つ」を入れているのは、AI に「紙が置かれたら動き出す」という始まり方も覚えさせるためです。

### 1枚ごとの記録（`record_episode`）
1/30 秒ごとに、次の順番で記録します。

1. **画像**：カメラの視点で描いた画像（紙・枠・押した後の印影つき）
2. **状態（`observation.state`）**：今の6つのモーターの角度（度）
3. **行動（`action`）**：次の 1/30 秒でめざす6つのモーターの目標角度（度）。モーターにはこの値を送り、1/30 秒のあいだ同じ値を保ちます。

3 の「行動を 1/30 秒保つ」というやり方は、学習したAIを動かすときと同じです。AI も 1/30 秒ごとに目標角度を1つ出して、モーターに送ります。お手本と本番の条件をそろえておくと、AIが真似しやすくなります。

角度の単位は「度」にしています。実機の LeRobot は設定によって別の単位（-100〜100 の割合）を使うことがあるので、実機のデータと混ぜるときに単位をそろえる必要があります。

In [ ]:
def plan_episode(aim):
    # 動きの予定表を作る（作れなければ None）
    reset_pose()
    q_home = sim_data.qpos[qadr].copy()
    q_above, e1, _ = solve_ik(aim + [0, 0, APPROACH_HEIGHT], sim_data.qpos)
    start = sim_data.qpos.copy()
    start[qadr] = q_above
    down, e2 = vertical_path(aim, PAPER_Z + APPROACH_HEIGHT, PAPER_Z - PRESS_DEPTH, start)
    if max(e1, e2) > 1e-3:
        return None
    q_press = down[-1]
    reset_pose()  # 逆運動学の計算で動かした状態を、初期姿勢に戻しておく
    return [  # (経路, 秒, 区間の種類)
        (np.array([q_home]), 0.5, "wait"),
        (np.array([q_home, q_above]), 1.2, "move"),
        (down, 0.8, "press"),
        (np.array([q_press]), 0.4, "press"),
        (down[::-1], 0.6, "lift"),
        (np.array([q_above, q_home]), 1.0, "move"),
        (np.array([q_home]), 0.3, "wait"),
    ]

def command_at(plan, t):
    # 時刻 t でめざす腕の関節角度と、区間の種類
    start = 0.0
    for path, duration, phase in plan:
        if t < start + duration:
            return pose_on_path(path, (t - start) / duration), phase
        start += duration
    path, _, phase = plan[-1]
    return path[-1], phase

def record_episode(frame_center, renderer):
    # お手本を1回分実行し、画像・状態・行動を1/30秒ごとに集める
    noise = rng.normal(0, DEMO_AIM_NOISE_MM / 1000, size=2) if DEMO_AIM_NOISE_MM > 0 else np.zeros(2)
    aim = frame_center + np.array([noise[0], noise[1], 0.0])
    plan = plan_episode(aim)
    if plan is None:
        return None

    total_time = sum(duration for _, duration, _ in plan)
    n_frames = int(round(total_time * fps))
    steps_per_frame = max(1, int(round(1 / (fps * sim_model.opt.timestep))))

    frames = []
    touch_xy, lowest_xy, min_z = None, None, np.inf
    for f in range(n_frames):
        t = f / fps

        # (1) 画像
        renderer.update_scene(sim_data, camera=cam)
        draw_overlay(renderer.scene, frame_center, touch_xy)
        image = renderer.render().copy()

        # (2) 状態：今のモーターの角度
        state = np.degrees(sim_data.qpos[motor_qadr]).astype(np.float32)

        # (3) 行動：次の1/30秒でめざす目標角度
        q_next, phase = command_at(plan, t + 1 / fps)
        set_arm_ctrl(q_next)
        action = np.degrees(sim_data.ctrl[:sim_model.nu]).astype(np.float32)

        frames.append({"image": image, "state": state, "action": action})

        # 物理計算を1/30秒分進める（印面の高さも見張る）
        for _ in range(steps_per_frame):
            mujoco.mj_step(sim_model, sim_data)
            p, _ = tip_pose(sim_data)
            if touch_xy is None and p[2] <= PAPER_Z + 0.001:
                touch_xy = p[:2].copy()
            if phase == "press" and p[2] < min_z:
                min_z, lowest_xy = p[2], p[:2].copy()

    if touch_xy is None or min_z > PAPER_Z + 0.002:
        return {"frames": frames, "success": False, "result": "紙に触れない", "offset_mm": np.nan}
    judged = judge_stamp(lowest_xy * 1000, frame_center[:2] * 1000)
    return {"frames": frames, "success": judged["success"],
            "result": "成功" if judged["success"] else "枠外",
            "offset_mm": judged["offset_mm"]}

print("記録の仕組みの準備完了")

## 7. 1回だけ試して動画で確認

本番の記録の前に、1回分だけお手本を作って動画で見ます。確認するのは次の3点です。

- アームが枠の真上へ動き、まっすぐ下りて押し、戻ってくるか
- 押したあと、枠の内側に赤い印影が出るか
- 結果が「成功」になっているか

In [ ]:
test_center = sample_frame_center()
with mujoco.Renderer(sim_model, height=IMG_H, width=IMG_W) as renderer:
    test_ep = record_episode(test_center, renderer)

print(f"結果: {test_ep['result']}  ずれ {test_ep['offset_mm']:.2f} mm  枚数 {len(test_ep['frames'])}")
media.show_video([fr["image"] for fr in test_ep["frames"]], fps=fps)

## 8. お手本をたくさん記録する（Google ドライブに1回ずつ保存）

紙の位置を毎回変えながら、お手本を `N_EPISODES` 回分記録します。

**途中で止まっても続きから再開できる**作りにしています。

- お手本を1回作るたびに、すぐ Google ドライブの `so101_stamp/raw_episodes` フォルダに保存します
  - `ep_000.mp4`：カメラ映像
  - `ep_000.npz`：関節の角度（状態と行動）、紙の位置、ずれ
- Colab の接続が切れても、ドライブに保存済みの分は消えません
- もう一度このセルを実行すると、保存済みの数を数えて、**続きの番号から**記録を再開します
- 再開したときに同じ紙の位置をくり返さないよう、乱数の種を保存済みの数に合わせて変えています

成功したお手本だけ保存します（失敗した動きを真似させないため）。

実行すると、最初に Google アカウントへのアクセス許可を求められるので、許可してください。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/so101_stamp")
RAW_DIR = DRIVE_DIR / "raw_episodes"
RAW_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR = Path("/content/tmp_episode")
TMP_DIR.mkdir(exist_ok=True)

def count_saved():
    # 映像と角度の両方がそろっているお手本の数
    return len([p for p in RAW_DIR.glob("ep_*.npz") if p.with_suffix(".mp4").exists()])

saved = count_saved()
print(f"ドライブに保存済み: {saved} 回 → {saved + 1} 回目から記録します")

rng = np.random.default_rng([SEED, saved])  # 再開のたびに別の紙の位置になるように
attempts = 0
with mujoco.Renderer(sim_model, height=IMG_H, width=IMG_W) as renderer:
    while saved < N_EPISODES and attempts < MAX_ATTEMPTS:
        attempts += 1
        center = sample_frame_center()
        ep = record_episode(center, renderer)
        if ep is None:
            print(f"試行{attempts:3d}: 届かないので飛ばします")
            continue
        if not ep["success"]:
            print(f"試行{attempts:3d}: {ep['result']}なので捨てます")
            continue

        name = f"ep_{saved:03d}"
        # いったん Colab の中に書き、できあがってからドライブへコピー（書きかけのファイルを残さない）
        media.write_video(TMP_DIR / f"{name}.mp4", [fr["image"] for fr in ep["frames"]], fps=fps)
        np.savez(TMP_DIR / f"{name}.npz",
                 state=np.array([fr["state"] for fr in ep["frames"]]),
                 action=np.array([fr["action"] for fr in ep["frames"]]),
                 frame_center_mm=center * 1000,
                 offset_mm=ep["offset_mm"])
        shutil.copy(TMP_DIR / f"{name}.mp4", RAW_DIR / f"{name}.mp4")
        shutil.copy(TMP_DIR / f"{name}.npz", RAW_DIR / f"{name}.npz")  # npz を最後に置く＝保存完了の印

        saved += 1
        print(f"試行{attempts:3d}: お手本 {saved}/{N_EPISODES} をドライブに保存（ずれ {ep['offset_mm']:.2f} mm）")

print(f"\nドライブに保存済み: {count_saved()} 回")
if count_saved() < N_EPISODES:
    print("まだ足りません。このセルをもう一度実行すると、続きから記録します")

## 9. 保存したお手本から、LeRobot のデータセットを作る

ドライブに保存したお手本を読み込み、LeRobot の学習データの形式（LeRobotDataset）にまとめます。物理計算や描画はしないので、数分で終わります。

- **`features`（データの中身の定義）**：何をどんな形で保存するかを LeRobot に伝えます
  - `observation.state`：6つのモーターの角度
  - `action`：6つのモーターの目標角度
  - `observation.images.front`：カメラ画像（480×640、カラー）。動画として圧縮して保存されます
- **`add_frame`**：1枚分のデータを追加します。`task` に指示の文を入れます
- **`save_episode`**：1回分のお手本を区切って保存します
- **`finalize`**：最後に必ず呼びます。呼ばないとデータセットのファイルが壊れたままになります

できあがったデータセットは、ドライブの `so101_stamp/so101_stamp_sim` にもコピーします。各回の紙の位置とずれの一覧（`so101_stamp_sim_episodes.csv`）も一緒に保存します。

接続が切れてこのセルが途中で止まった場合は、「1. インストール」から「4. 道具」まで実行し直してから、このセルだけ実行すれば大丈夫です（8 はやり直さなくて構いません）。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/so101_stamp")
RAW_DIR = DRIVE_DIR / "raw_episodes"

try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

episode_files = sorted(p for p in RAW_DIR.glob("ep_*.npz") if p.with_suffix(".mp4").exists())
print(f"使うお手本: {len(episode_files)} 回")

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)  # 前回の作りかけが残っていたら消す

n_motor = len(MOTOR_NAMES)
features = {
    "observation.state": {"dtype": "float32", "shape": (n_motor,), "names": MOTOR_NAMES},
    "action": {"dtype": "float32", "shape": (n_motor,), "names": MOTOR_NAMES},
    "observation.images.front": {"dtype": "video", "shape": (IMG_H, IMG_W, 3),
                                 "names": ["height", "width", "channels"]},
}
dataset = LeRobotDataset.create(
    repo_id=REPO_ID, fps=fps, features=features, root=DATASET_ROOT,
    robot_type="so101_follower", use_videos=True,
)

def save_episode_compat(ds):
    # LeRobot のバージョンによる書き方の違いを吸収する
    try:
        ds.save_episode()
    except TypeError:
        ds.save_episode(task=TASK)

log = []
for i, npz_path in enumerate(episode_files):
    data = np.load(npz_path)
    images = media.read_video(npz_path.with_suffix(".mp4"))
    n = min(len(images), len(data["state"]))
    for k in range(n):
        dataset.add_frame({
            "observation.state": data["state"][k].astype(np.float32),
            "action": data["action"][k].astype(np.float32),
            "observation.images.front": np.asarray(images[k], dtype=np.uint8),
            "task": TASK,
        })
    save_episode_compat(dataset)
    c = data["frame_center_mm"]
    log.append({"episode": i, "frame_x_mm": float(c[0]), "frame_y_mm": float(c[1]),
                "offset_mm": float(data["offset_mm"])})
    print(f"{i + 1}/{len(episode_files)} {npz_path.name} を追加")

dataset.finalize()

log_path = DATASET_ROOT.parent / "so101_stamp_sim_episodes.csv"
with open(log_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["episode", "frame_x_mm", "frame_y_mm", "offset_mm"])
    writer.writeheader()
    writer.writerows(log)

shutil.copytree(DATASET_ROOT, DRIVE_DIR / DATASET_ROOT.name, dirs_exist_ok=True)
shutil.copy(log_path, DRIVE_DIR / log_path.name)
print(f"\nデータセットを作り、ドライブに保存しました: {DRIVE_DIR / DATASET_ROOT.name}")

## 10. データセットの中身を確認

保存したデータセットを、学習のときと同じ方法で読み込み直して確認します。

- **回数と枚数**：お手本の回数が 50、全部の枚数が 7200（50回 × 144枚）なら正常です
- **1枚目の中身**：画像・状態・行動などが、どんな形で入っているか
- **画像**：1枚目の画像を表示します。学習のときに AI が見る画像です
- **グラフ**：最後のお手本で、6つのモーターの「目標（action）」と「実際（state）」の角度の動きを比べます。実際の角度が目標を少し遅れて追いかけていれば正常です

In [ ]:
ds = LeRobotDataset(REPO_ID, root=DATASET_ROOT)
print(f"お手本の回数: {ds.num_episodes}")
print(f"全部の枚数:   {ds.num_frames}")
print(f"1秒あたり:    {ds.fps} 枚")

sample = ds[0]
print("\n1枚目の中身:")
for k, v in sample.items():
    print(f"  {k:<28} {tuple(v.shape) if hasattr(v, 'shape') else v}")

img = sample["observation.images.front"]
media.show_image(img.permute(1, 2, 0).numpy(), title="学習で使う画像（1枚目）")

last = np.load(episode_files[-1])
states, actions = last["state"], last["action"]
t = np.arange(len(states)) / fps
fig, axes = plt.subplots(len(MOTOR_NAMES), 1, figsize=(8, 1.8 * len(MOTOR_NAMES)), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(t, actions[:, i], "--", label="action")
    ax.plot(t, states[:, i], label="state")
    ax.set_ylabel(f"{MOTOR_NAMES[i]}\n[deg]", fontsize=8)
    ax.grid(alpha=0.3)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("time [s]")
plt.tight_layout()
plt.show()